# Lab 2 — Data as a System

**MA553 · Session 3**

Name:

---

Six monthly files of New York taxi trips, about 20 million rows, must be turned
into a table of total charges by borough. A colleague has written a script that
does this. It runs without error and prints a table. Your task is to decide
whether the numbers are right, and to produce a figure that can be defended.

**Part A** introduces the methods: reading only what is needed, processing one
file at a time, combining per-file results, and checking the file structure.

**Part B** is the audit. The report script contains five faults. Each changes
the total, each can be found with a check from the lecture, and each needs its
own correction.

Work through the cells in order. Five cells are marked **Write this yourself**:
A2, A4, A5, B5 and B7. Each says which variables to create; later cells use
those names.

# Part A — Reading and processing the files

## The data

Run `python get_data.py` in this folder once, before starting. It downloads
six monthly files (October 2024 to March 2025) and the city's zone table into
`data/`. About 350 MB.

Each row of a monthly file is one trip. The columns used in this lab:

| Column | Meaning |
|---|---|
| `tpep_pickup_datetime` | when the trip began, New York local time |
| `PULocationID` | pickup zone number, 1 to 265 |
| `total_amount` | what the passenger was charged |
| `trip_distance` | miles, from the meter |

The zone table (`data/taxi_zone_lookup.csv`) gives each zone number its borough
and name.

Run this.

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("data")
FILES = sorted(DATA.glob("yellow_tripdata_*.parquet"))
ZONES = pd.read_csv(DATA / "taxi_zone_lookup.csv")

print(f"{len(FILES)} monthly files:")
for f in FILES:
    print(f"  {f.name:34s} {f.stat().st_size / 1e6:6.1f} MB")
print(f"\nZone table: {len(ZONES)} rows, columns {list(ZONES.columns)}")

If fewer than six files are listed, run `get_data.py` again; it skips files
already downloaded.

## A1 — Define the calculation

Before reading anything at scale, fix what is being computed. For this lab:

> **A trip counts for a month if its pickup date falls in that month.
> The quantity is the sum of `total_amount`, with negative amounts kept.**

The January file is used for the examples. Run this to see why the definition
matters: four readings of "total charges for January" give four numbers.

In [ ]:
jan = pd.read_parquet(DATA / "yellow_tripdata_2025-01.parquet",
                      columns=["tpep_pickup_datetime", "total_amount", "fare_amount"])
in_january = jan["tpep_pickup_datetime"].dt.strftime("%Y-%m") == "2025-01"

print(f"Rows in the file:                                   {len(jan):>12,}")
print(f"total_amount, every row:                        ${jan['total_amount'].sum():>15,.2f}")
print(f"fare_amount, every row:                         ${jan['fare_amount'].sum():>15,.2f}")
print(f"total_amount, January pickups only:             ${jan.loc[in_january, 'total_amount'].sum():>15,.2f}")
print(f"total_amount, January, negatives excluded:      ${jan.loc[in_january & (jan['total_amount'] >= 0), 'total_amount'].sum():>15,.2f}")
print(f"\nTrips in the file that did not begin in January: {(~in_january).sum()}")

Under the definition above, the January figure is the third line. Keep it in
view; every later number is compared with it.

**Answer here.** Row 4 is larger than row 3 although it excludes rows. Explain
in one sentence why.

*Your answer:*

## A2 — The same table in two file formats

**Write this yourself.**

The city publishes the data as Parquet. Measure what CSV would have cost.

What to do:

1. Read the January file completely with `pd.read_parquet` and write it to
   `data/january.csv` with `.to_csv(..., index=False)`. This takes a minute.
2. Time one read of `data/january.csv` with `pd.read_csv`, and one read of the
   Parquet file with `pd.read_parquet`. Use `time.time()` before and after each
   read. Store the two durations as `csv_seconds` and `parquet_seconds`.
3. Store the two file sizes in megabytes as `csv_mb` and `parquet_mb`
   (`Path.stat().st_size / 1e6`).
4. Print all four, labelled.

Expected: the CSV is several times larger and several times slower to read.
The exact ratio depends on your machine.

In [ ]:
# Your code here.


pandas may print a warning that one column "has mixed types". That is the
third row of the lecture's comparison table: a CSV records no column types, so
the reader guesses them, and here it guessed inconsistently.

**Answer here.** The CSV holds the same rows and columns. Name one reason it is
larger and one reason it is slower to read.

*Your answer:*

## A3 — Reading fewer columns

The calculation needs three of the twenty columns. Parquet stores each column
separately, so the other seventeen need not be read. Run this.

In [ ]:
NEEDED = ["tpep_pickup_datetime", "PULocationID", "total_amount"]

start = time.time()
full = pd.read_parquet(DATA / "yellow_tripdata_2025-01.parquet")
full_seconds = time.time() - start
full_mb = full.memory_usage(deep=True).sum() / 1e6

start = time.time()
part = pd.read_parquet(DATA / "yellow_tripdata_2025-01.parquet", columns=NEEDED)
part_seconds = time.time() - start
part_mb = part.memory_usage(deep=True).sum() / 1e6

print(f"All 20 columns: {full_seconds:4.1f} s to read, {full_mb:5.0f} MB in memory")
print(f"3 columns:      {part_seconds:4.1f} s to read, {part_mb:5.0f} MB in memory")
del full

A 59 MB file became a table several times larger in memory: the file is
encoded and compressed, the table is not. Six months of all columns would need
about 3 GB. Six months of three columns need about 500 MB, and we will not hold
even that at once.

## A4 — Six months, one file at a time

**Write this yourself.**

Compute the number of trips and the total charges for every day in the six
months, without loading two files at once.

What to do:

1. Create an empty list `parts`.
2. Loop over `path` in `FILES`. For each file:
   - Read only `NEEDED` columns.
   - Work out the file's month from its name: `path.stem[-7:]` gives
     `"2024-10"` and so on.
   - Keep only trips whose pickup month equals the file's month
     (`df["tpep_pickup_datetime"].dt.strftime("%Y-%m") == month`). Print the
     file name and how many rows were dropped.
   - Group the kept rows by pickup *date* (`df["tpep_pickup_datetime"].dt.date`)
     and compute two columns: `trips` (the count) and `total_amount` (the sum).
     Add a column `source_file` holding `path.name`.
   - Append the result to `parts`.
3. After the loop, `daily = pd.concat(parts)`. Make sure the date is a column
   called `date`, not the index.
4. Print the number of days in `daily`, the total number of trips, and the grand
   total of `total_amount`.

Expected with six files: about 180 days, about 20 million trips, a grand total
near $500 million, and 20 to 60 rows dropped per file.

In [ ]:
# Your code here.


Because each file was filtered to its own month, no date appears in two files
and the per-day rows stack without adding. Keep `daily`; Part B compares the
report against it.

**Answer here.** `daily` holds a few hundred rows. From it, which of the
following can be computed exactly for the six months, and which cannot: the
total charges; the mean charge per trip; the median charge per trip? Say why
for each.

*Your answer:*

## A5 — A check on the file structure

**Write this yourself.**

The files are not identical. Write a check that catches a change in the
columns before any calculation uses them.

What to do:

1. Write a function `check_columns(path, required)` that reads only the
   column names of the Parquet file (`pd.read_parquet(path).columns` reads the
   whole file; use `pyarrow.parquet.read_schema(path).names` instead) and
   raises `ValueError` naming any column in `required` that is absent. It
   returns the list of column names when all are present.
2. Call it on every file in `FILES` with `required=NEEDED`. Nothing should be
   raised.
3. Call it on `FILES[0]` (October 2024) with
   `required=["cbd_congestion_fee"]` inside a `try`/`except ValueError` and
   print the error message.
4. Print the number of columns in each file.

Expected: the three needed columns are present everywhere; the October file has
19 columns and no `cbd_congestion_fee`; files from January 2025 have 20.

In [ ]:
# Your code here.


A schema check costs nothing and turns a silent change into a loud one. The
report script in Part B has no such check.

# Part B — The audit

## The situation

`monthly_report.py` in this folder computes total charges by borough for the
six months and prints a grand total. It was written to the same definition as
A1: pickup date decides the month, `total_amount` is the quantity, negative
amounts stay in.

Read the script before running it. It is short.

## B1 — Predict first

**Answer here.** From A4 you know the six-month total under the definition.
Before running the script, write down that number, and list anything in the
script that you expect to move the result away from it.

*Your answer:*

## B2 — Run the report

In [ ]:
from monthly_report import run_report, charges_by_borough

reported = run_report()
correct_total = daily["total_amount"].sum()
print(f"\nSix-month total from A4:  {correct_total:,.2f}")
print(f"Difference:               {reported.sum() - correct_total:,.2f}")

The report is low by millions of dollars. Nothing in its output says so;
the comparison with A4 is what reveals it.

The remaining cells find the faults one at a time. To measure each one, we
apply the script's logic to a single file, the January file, and change one
thing at a time. Run this to set up the January trips and the two zone tables.

In [ ]:
trips = pd.read_parquet(DATA / "yellow_tripdata_2025-01.parquet")
zones_published = pd.read_csv(DATA / "taxi_zone_lookup.csv")
zones_revised = pd.read_csv("taxi_zone_lookup_revised.csv")

in_january = trips["tpep_pickup_datetime"].dt.strftime("%Y-%m") == "2025-01"
january_total = trips.loc[in_january, "total_amount"].sum()

script_january = charges_by_borough(trips, zones_revised).sum()
print(f"January under the definition:  {january_total:>15,.2f}")
print(f"January from the script:       {script_january:>15,.2f}")
print(f"Difference:                    {script_january - january_total:>15,.2f}")

## B3 — Fault one: `dropna()` removes trips, not blanks

The script's first line is `trips.dropna()`: keep only rows with no missing
value in any column. Run this to see what that removes.

In [ ]:
missing_any = trips.isna().any(axis=1)
print(f"Rows with at least one missing value: {missing_any.sum():,} of {len(trips):,} "
      f"({missing_any.mean():.1%})")
print(f"Their total_amount:                   ${trips.loc[missing_any, 'total_amount'].sum():,.2f}")
print("\nMissing values by column:")
print(trips.isna().sum()[trips.isna().sum() > 0].to_string())

**Answer here.** The missing values are in passenger count, rate code and two
surcharge columns. The calculation uses none of them. State in one sentence
what `dropna()` assumed about those trips, and whether the data supports it.

*Your answer:*

## B4 — Fault two: an exclusion the definition does not allow

The script drops trips with `total_amount < 0` as "refunds and reversals".
The definition keeps them. Run this.

In [ ]:
negative = trips["total_amount"] < 0
print(f"Trips with a negative total: {negative.sum():,}")
print(f"Their sum:                   ${trips.loc[negative, 'total_amount'].sum():,.2f}")
print(f"Effect of excluding them:    +${-trips.loc[negative, 'total_amount'].sum():,.2f}")

**Answer here.** A negative charge may be a refund, a reversal, or an error.
Whether to exclude it is a decision about the definition, not a correction.
Where should that decision be recorded, and what would the report have to say
if it kept the exclusion?

*Your answer:*

## B5 — Fault three: the zone table attaches twice

**Write this yourself.**

The script joins to `taxi_zone_lookup_revised.csv`, a copy of the zone table
that someone has edited. Check it before trusting the join.

What to do:

1. Print how many rows each zone table has, and how many duplicated
   `LocationID` values each has (`.duplicated().sum()`).
2. Print the rows of `zones_revised` whose `LocationID` is duplicated
   (`zones_revised[zones_revised["LocationID"].duplicated(keep=False)]`).
3. Join `trips` to each zone table in turn with
   `trips.merge(z[["LocationID", "Borough"]], left_on="PULocationID",
   right_on="LocationID", how="left")`. For each join print the row count and
   the sum of `total_amount`. Store the two row counts as `rows_published` and
   `rows_revised`.
4. Print how many January trips were picked up in the duplicated zone.

Expected: the published table has 265 rows and no duplicates; the revised one
has 266 rows and one duplicated zone; the join to it has exactly that zone's
trip count in extra rows, and the total rises by exactly those trips' charges.

In [ ]:
# Your code here.


**Answer here.** The join raised the total, and so did excluding negative
totals, while `dropna()` lowered it. Explain why the grand total in B2 can be
wrong by a smaller amount than the largest single fault.

*Your answer:*

## B6 — Fault four: grouping drops trips with no borough

Zone 265 ("Outside of NYC") has no borough in the zone table. The script groups
by borough with the default setting, which leaves those trips out of every
group. Run this.

In [ ]:
by_borough = joined_published.groupby("Borough")["total_amount"].agg(["size", "sum"])
by_borough_all = joined_published.groupby("Borough", dropna=False)["total_amount"].agg(["size", "sum"])

print(f"Sum of borough groups, default:       ${by_borough['sum'].sum():,.2f}  ({by_borough['size'].sum():,} trips)")
print(f"Sum of borough groups, dropna=False:  ${by_borough_all['sum'].sum():,.2f}  ({by_borough_all['size'].sum():,} trips)")
print(f"January total, all trips:             ${trips['total_amount'].sum():,.2f}  ({len(trips):,} trips)")
print("\nThe dropped group:")
print(by_borough_all[by_borough_all.index.isna()].to_string())

Zone 264 ("Unknown") is unaffected: its borough is the word `Unknown`, which
is a value, so it forms a group. Only a genuinely missing borough is dropped.

## B7 — Fault five: the file name is not the month

**Write this yourself.**

The script never looks at the pickup date. Measure what that does across all
six files.

What to do:

1. Create an empty list `strays`.
2. Loop over `path` in `FILES`. Read only `tpep_pickup_datetime` and
   `total_amount`. Compute `month = path.stem[-7:]` and the pickup month of
   each row as in A4. Append a tuple `(path.name, count, amount)` to `strays`,
   where `count` is the number of rows whose pickup month differs from the
   file's month and `amount` is the sum of their `total_amount`.
3. Turn `strays` into a DataFrame with columns `file`, `trips`, `total_amount`
   and print it.
4. Print the total number of stray trips and their total charges.

Expected: a few dozen stray trips per file, a few thousand dollars in all.
This fault is the smallest of the five in this data. It is not always.

In [ ]:
# Your code here.


**Answer here.** Two of the stray trips in the January file began on
1 February. If the February file also contains them, what does the script do
with them, and what does the A4 procedure do?

*Your answer:*

## B8 — The correction table

Fill this in for the January file, using the numbers printed above. "Effect"
is the change in the January total when that fault alone is corrected, with a
sign.

| Fault | Which problem from the lecture | Check that catches it | Effect on the January total |
|---|---|---|---|
| 1. `dropna()` | | | |
| 2. Negative totals excluded | | | |
| 3. Revised zone table | | | |
| 4. `groupby` default | | | |
| 5. No pickup-date filter | | | |

Then run this. It applies the five corrections together and compares with the
definition.

In [ ]:
def charges_by_borough_corrected(trips, zones, month):
    assert zones["LocationID"].is_unique, "zone table has duplicated zone numbers"
    in_month = trips["tpep_pickup_datetime"].dt.strftime("%Y-%m") == month
    trips = trips[in_month]
    with_borough = trips.merge(zones[["LocationID", "Borough"]],
                               left_on="PULocationID", right_on="LocationID", how="left")
    assert len(with_borough) == len(trips), "join changed the row count"
    return with_borough.groupby("Borough", dropna=False)["total_amount"].sum()


corrected = charges_by_borough_corrected(trips, zones_published, "2025-01")
print(corrected.map("{:>16,.2f}".format).to_string())
print(f"\nSum of groups:               {corrected.sum():,.2f}")
print(f"January under the definition: {january_total:,.2f}")

## B9 — The report

Write three things, one short paragraph each.

**1. The result.** The six-month total charges by borough, computed with the
corrected function over all six files (write that loop here, following A4),
and the grand total. State the definition it answers.

*Your answer:*

In [ ]:
# Your code here.


**2. One conclusion.** What the comparison between the script's figure and the
corrected figure establishes about the script, in one sentence, with the
numbers.

*Your answer:*

**3. One limitation.** Something the corrected figure still does not establish,
and what would be needed to establish it.

*Your answer:*

## Before you submit

- Restart the kernel and run every cell from top to bottom. A2 and A4 take a
  few minutes; that is expected.
- Delete `data/january.csv` afterwards. Do not submit the `data/` folder.
- Check that every **Answer here** cell is filled in, and the correction table.
- If you used an AI assistant, add a short note at the end saying what it
  produced and what you changed. You must be able to explain every line.